# NHL Data Pipeline
This notebook pulls NHL standings data, cleans it for analysis, and saves outputs to CSV and SQLite.

## Section 1: Imports
Load required libraries for API requests, data manipulation, and database export.

In [14]:
# Standard library imports
import sqlite3

# Third-party imports
import pandas as pd
import requests

## Section 2: NHL API Connection Test
Verifying connectivity to the NHL API before building the data pipeline. A 200 status code confirms the connection is live.

In [15]:
# API connection test
response = requests.get("https://api-web.nhle.com/v1/standings/now")
status = response.status_code
print(f"API status: {status} ({'OK' if status == 200 else 'FAILED'})")

API status: 200 (OK)


## Section 3: Data Pull
Request standings data from the API response payload and shape it into a raw dataframe for downstream processing.

In [16]:
# Parse standings payload from API response
standings = response.json()["standings"]
print(f"Pulled standings for {len(standings)} teams")

Pulled standings for 32 teams


In [17]:
# Quick field check (single exploratory output)
print("Available fields:", sorted(standings[0].keys()))

Available fields: ['clinchIndicator', 'conferenceAbbrev', 'conferenceHomeSequence', 'conferenceL10Sequence', 'conferenceName', 'conferenceRoadSequence', 'conferenceSequence', 'date', 'divisionAbbrev', 'divisionHomeSequence', 'divisionL10Sequence', 'divisionName', 'divisionRoadSequence', 'divisionSequence', 'gameTypeId', 'gamesPlayed', 'goalAgainst', 'goalDifferential', 'goalDifferentialPctg', 'goalFor', 'goalsForPctg', 'homeGamesPlayed', 'homeGoalDifferential', 'homeGoalsAgainst', 'homeGoalsFor', 'homeLosses', 'homeOtLosses', 'homePoints', 'homeRegulationPlusOtWins', 'homeRegulationWins', 'homeTies', 'homeWins', 'l10GamesPlayed', 'l10GoalDifferential', 'l10GoalsAgainst', 'l10GoalsFor', 'l10Losses', 'l10OtLosses', 'l10Points', 'l10RegulationPlusOtWins', 'l10RegulationWins', 'l10Ties', 'l10Wins', 'leagueHomeSequence', 'leagueL10Sequence', 'leagueRoadSequence', 'leagueSequence', 'losses', 'otLosses', 'placeName', 'pointPctg', 'points', 'regulationPlusOtWinPctg', 'regulationPlusOtWins', 'r

In [18]:
# Convert standings payload into raw dataframe
df_standings = pd.DataFrame(standings)
print(df_standings.shape)
df_standings.head()

(32, 81)


,clinchIndicator,conferenceAbbrev,conferenceHomeSequence,conferenceL10Sequence,conferenceName,conferenceRoadSequence,conferenceSequence,date,divisionAbbrev,divisionHomeSequence,...,streakCount,teamName,teamCommonName,teamAbbrev,teamLogo,ties,waiversSequence,wildcardSequence,winPctg,wins
0,p,W,1,2,Western,1,1,2026-04-17,C,1,...,3,"{'default': 'Colorado Avalanche', 'fr': 'Avala...",{'default': 'Avalanche'},{'default': 'COL'},https://assets.nhle.com/logos/nhl/svg/COL_ligh...,0,32,0,0.670732,55
1,z,E,1,2,Eastern,3,1,2026-04-17,M,1,...,1,"{'default': 'Carolina Hurricanes', 'fr': 'Hurr...",{'default': 'Hurricanes'},{'default': 'CAR'},https://assets.nhle.com/logos/nhl/svg/CAR_ligh...,0,31,0,0.646341,53
2,x,W,2,3,Western,2,2,2026-04-17,C,2,...,5,"{'default': 'Dallas Stars', 'fr': 'Stars de Da...",{'default': 'Stars'},{'default': 'DAL'},https://assets.nhle.com/logos/nhl/svg/DAL_ligh...,0,30,0,0.609756,50
3,y,E,3,6,Eastern,4,2,2026-04-17,A,2,...,1,"{'default': 'Buffalo Sabres', 'fr': 'Sabres de...",{'default': 'Sabres'},{'default': 'BUF'},https://assets.nhle.com/logos/nhl/svg/BUF_ligh...,0,29,0,0.609756,50
4,x,E,5,12,Eastern,2,3,2026-04-17,A,3,...,1,"{'default': 'Tampa Bay Lightning', 'fr': 'Ligh...",{'default': 'Lightning'},{'default': 'TBL'},https://assets.nhle.com/logos/nhl/svg/TBL_ligh...,0,28,0,0.609756,50


## Section 4: Data Cleaning
Flatten nested fields, remove unused columns, and prepare a clean standings table for analysis and storage.

In [19]:
# Extract clean team name and abbreviation from nested dicts
df_standings['team_name'] = df_standings['teamCommonName'].apply(lambda x: x['default'])
df_standings['team_abbrev'] = df_standings['teamAbbrev'].apply(lambda x: x['default'])
df_standings['place_name'] = df_standings['placeName'].apply(lambda x: x['default'])

# Drop columns we don't need
df_standings = df_standings.drop(columns=['teamName', 'teamCommonName', 'teamAbbrev', 'teamLogo', 'placeName'])

# Verify it looks clean
df_standings[['team_name', 'team_abbrev', 'points', 'wins', 'losses', 'goalFor', 'goalAgainst']].head()

,team_name,team_abbrev,points,wins,losses,goalFor,goalAgainst
0,Avalanche,COL,121,55,16,302,203
1,Hurricanes,CAR,113,53,22,296,240
2,Stars,DAL,112,50,20,279,226
3,Sabres,BUF,109,50,23,288,241
4,Lightning,TBL,106,50,26,290,231


## Section 5: Save Outputs
Write both raw and cleaned standings to CSV, then persist cleaned standings to the SQLite database.

In [20]:
# Save raw and cleaned CSV outputs
pd.DataFrame(standings).to_csv('../data/raw/standings_now.csv', index=False)
df_standings.to_csv('../data/processed/standings_clean.csv', index=False)

# Save cleaned standings to SQLite database
conn = sqlite3.connect('../database/nhl.db')
df_standings.to_sql('standings_clean', conn, if_exists='replace', index=False)
conn.close()

print("Saved raw CSV, cleaned CSV, and SQLite table: standings_clean")

Saved raw CSV, cleaned CSV, and SQLite table: standings_clean
